# Task 1 ensemble sweep

Which of the trained sub-models should the Task 1 blend combine, and with which blender settings?

**Run after the training:** `submission/task1_ensemble.ipynb` stores every sub-model in `trained_models/task1/`. This notebook only
uses the stored validation predictions, so nothing is retrained. Sub-models that are not stored yet are left out and printed.
Run All runs every stage. Run time: a ridge blend takes about a second; most of the run time are the LightGBM blenders.

**How a composition is scored:** the out-of-fold MSE on the 10 % validation ratings (lower is better), the way the submitted
rating blender (ridge / LightGBM) is scored, averaged over 3 different splits of the validation ratings into folds.
Two compositions are compared rating by rating: change +- standard error. A change within about two standard errors is noise.

1. **Single models** and the blend of all
2. **Drop one:** the blend without each model -> how much it adds on top of all the others
3. **Backward elimination:** drop the least useful model again and again, as long as the blend stays within `tolerance` of the
   blend of all. Bias model and item kNN only together: the blend uses their difference (the kNN's neighbourhood correction).
4. **Blender settings** on the selected models, compared with the current blender
5. **Recommendation** + report in `output/task1_ensemble/ensemble_<start time>.json`

**Apply the result:** copy the printed `TASK1_BLEND_MODEL_FLAGS` into `lib/config.py` and set `TASK1_SUBMITTED_BLENDER` to the printed
blender config (with the printed setting changes, if any),
then rerun the blend cells of `task1_ensemble.ipynb`.

Why drop models at all: ridge gives an unhelpful model a weight near 0, so a larger blend rarely hurts. Dropping a model mainly
saves its training time (Bayesian FM, rating transformer) and keeps the ensemble simpler; `tolerance` is the accuracy you are
willing to give up for that.

In [ ]:
%load_ext autoreload
# --full: reload whole modules. The default only re-runs the changed lines of a module, so a setting computed from an
# edited one (e.g. TASK1_BLENDED_MODELS from TASK1_BLEND_MODEL_FLAGS in lib/config.py) would keep its old value
%autoreload 2 --full

# ensemble_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
from dataclasses import replace

from ensemble_search import (RatingBlendScorer, backward_elimination, blend_units, blender_comparison, drop_one_table,
                             load_stored_sub_models, print_recommendation, save_ensemble_report, single_model_table)

import assignment1
import lib.config as model_configs        # settings read when a cell runs -> edits of config.py apply without a restart

In [ ]:
# candidates: every sub-model with a flag in lib/config.py (True or False -> a left-out model can come back)
candidate_models = tuple(model_configs.TASK1_BLEND_MODEL_FLAGS)
linked_models = [("bias_model", "item_knn")]    # only dropped together (the blend uses their difference)
fold_seeds = (0, 1, 2)
tolerance = 0.0002                              # MSE the selected blend may lose against the blend of all (~2 standard errors)

# stage 4: the first entry is the reference the others are compared with
blender_candidates = {
    "ridge alpha 1 (submitted)": model_configs.TASK1_RIDGE_BLENDER,
    "ridge alpha 0.1": replace(model_configs.TASK1_RIDGE_BLENDER, ridge_alpha=0.1),
    "ridge alpha 10": replace(model_configs.TASK1_RIDGE_BLENDER, ridge_alpha=10.0),
    "ridge alpha 100": replace(model_configs.TASK1_RIDGE_BLENDER, ridge_alpha=100.0),
    "LightGBM + counts (lib/config.py)": model_configs.TASK1_LIGHTGBM_BLENDER,
    "LightGBM without counts": replace(model_configs.TASK1_LIGHTGBM_BLENDER, use_count_features=False),
    "LightGBM + counts, 15 leaves": replace(model_configs.TASK1_LIGHTGBM_BLENDER, lightgbm_num_leaves=15),
    "LightGBM + counts, 63 leaves": replace(model_configs.TASK1_LIGHTGBM_BLENDER, lightgbm_num_leaves=63),
}

In [ ]:
task1 = assignment1.load_task1()
sub_model_results = load_stored_sub_models("task1", candidate_models)
stored_models = tuple(sub_model_results)
units = blend_units(stored_models, linked_models)
scorer = RatingBlendScorer(task1, sub_model_results, fold_seeds)
report = {"task": "task1", "stored_models": stored_models, "missing_models": [m for m in candidate_models if m not in stored_models],
          "fold_seeds": fold_seeds, "tolerance": tolerance, "base_blender": model_configs.TASK1_RIDGE_BLENDER}
print(f"{len(stored_models)} stored sub-models, {len(units)} units: {units}")

## 1. Single models and the blend of all
The zero models predict P(0), not a rating -> no MSE of their own.

In [ ]:
report["single"] = single_model_table(scorer, model_configs.TASK1_RIDGE_BLENDER, stored_models)

## 2. Drop one
Change of the blend without each model (positive MSE change = the blend gets worse without it = the model helps). Most useful first.

In [ ]:
report["drop_one"] = drop_one_table(scorer, model_configs.TASK1_RIDGE_BLENDER, units)

## 3. Backward elimination
Every step drops the model whose removal hurts least; it stops before the blend would lose more than `tolerance`
against the blend of all models.

In [ ]:
elimination = backward_elimination(scorer, model_configs.TASK1_RIDGE_BLENDER, units, tolerance)
report["elimination"] = elimination

## 4. Blender settings on the selected models
Each blender compared with the first entry (the current one). Another blender is only recommended if it beats it by more than
two standard errors.

In [ ]:
blenders = blender_comparison(scorer, blender_candidates, elimination["selected_models"])
report["blenders"] = blenders

## 5. Recommendation
Copy into `lib/config.py` (Task 1 section), then rerun the blend cells of `task1_ensemble.ipynb`.

In [ ]:
report["recommendation"] = print_recommendation(scorer, blender_candidates, elimination, blenders, "TASK1_BLEND_MODEL_FLAGS",
                                                model_configs.TASK1_BLEND_MODEL_FLAGS, stored_models,
                                                {"TASK1_RIDGE_BLENDER": model_configs.TASK1_RIDGE_BLENDER, "TASK1_LIGHTGBM_BLENDER": model_configs.TASK1_LIGHTGBM_BLENDER})
if report["missing_models"]:
    print(f"(not part of this search, not stored yet: {', '.join(report['missing_models'])})")
save_ensemble_report("task1", report)